# TrialConfigExamples

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `signal`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/TrialConfigExamples.md`


Notebook source link: [TrialConfigExamples.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/TrialConfigExamples.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "TrialConfigExamples"
FAMILY = "signal"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"TrialConfigExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"TrialConfigExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"TrialConfigExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"TrialConfigExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "tc1 = TrialConfig({'Force','f_x'},2000,[.1 .2],-1,2);",
    "tc2 = TrialConfig({'Position','x'},2000,[.1 .2],-1,2);",
    "tcc = ConfigColl({tc1,tc2});"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for TrialConfigExamples.")


In [ ]:
# TrialConfigExamples: create and inspect trial configurations.
from nstat.compat.matlab import TrialConfig, ConfigColl

tc1 = TrialConfig(covariateLabels=["Force", "f_x"], Fs=2000.0, fitType="poisson", name="ForceX")
tc2 = TrialConfig(covariateLabels=["Position", "x"], Fs=2000.0, fitType="poisson", name="PositionX")
tcc = ConfigColl([tc1, tc2])

config_names = tcc.getConfigNames()
cfg1 = tcc.getConfig(1)
cfg2 = tcc.getConfig("PositionX")
sample_rates = np.array([cfg.sample_rate_hz for cfg in tcc.getConfigs()], dtype=float)

fig, ax = plt.subplots(1, 1, figsize=(7.6, 4.2))
ax.bar(config_names, sample_rates, color=["tab:blue", "tab:orange"])
ax.set_ylabel("sample rate [Hz]")
ax.set_title(f"{TOPIC}: TrialConfig summary")
plt.tight_layout()
plt.show()

assert cfg1.getSampleRate() == 2000.0
assert cfg2.getFitType() == "poisson"

CHECKPOINT_METRICS = {
    "num_configs": float(len(tcc.getConfigs())),
    "sample_rate_hz": float(np.mean(sample_rates)),
}
CHECKPOINT_LIMITS = {
    "num_configs": (2.0, 2.0),
    "sample_rate_hz": (2000.0, 2000.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
